In [ ]:
!pip install -q transformers torch faiss-cpu Pillow tqdm h5py

In [ ]:
import os, time, numpy as np, torch, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor
import faiss, h5py, pickle

# ---------------------------------------------------------------------------
# Universal secret getter – works on Kaggle (Secrets / Private Dataset) and
# locally via .env
# ---------------------------------------------------------------------------
def get_secret(key: str, fallback_path: str = None, default=None) -> str:
    # 1. Try Kaggle Secrets (manual run)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        pass

    # 2. Try private dataset fallback (API push)
    if fallback_path and os.path.exists(fallback_path):
        return open(fallback_path).read().strip()

    # 3. Try .env file (local dev)
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    value = os.environ.get(key, default)
    if value is None:
        raise EnvironmentError(f"Secret '{key}' not found in Kaggle Secrets, private dataset, or .env")
    return value

# Get HF_TOKEN (3-tier fallback)
hf_token = get_secret("HF_TOKEN", fallback_path="/kaggle/input/my-hf-secrets/HF_TOKEN.txt")
print("✓ HF_TOKEN loaded")

# ---------------------------------------------------------------------------
# Dataset path detection (same convention as other notebooks)
# ---------------------------------------------------------------------------
paths = [
    "/kaggle/input/chetankv/dogs-cats-images/dataset/test_set/cats",
    "/kaggle/input/dogs-cats-images/dataset/test_set/cats",
    "/kaggle/input/datasets/chetankv/dogs-cats-images/dataset/test_set/cats"
]
DATASET_PATH = next((p for p in paths if os.path.exists(p)), None)
if not DATASET_PATH:
    raise FileNotFoundError("Dataset not found")

# ---------------------------------------------------------------------------
# Model / output config
# NOTE: We reuse the SAME model as dino-v3-embed (facebook/dinov3-vitb16-pretrain-lvd1689m)
# but extract PATCH TOKENS instead of the CLS pooler_output.
# Patch mean-pool dim == 768 (same as CLS), so the same FAISS index type works.
# ---------------------------------------------------------------------------
MODEL_ID    = "facebook/dinov3-vitb16-pretrain-lvd1689m"
EMBEDDING_DIM = 768   # hidden size of ViT-B
OUTPUT_DIR  = "/kaggle/working"

# ---------------------------------------------------------------------------
# Adaptive GPU / CPU configuration
# ---------------------------------------------------------------------------
gpu_name = ""
if torch.cuda.is_available():
    num_gpus  = torch.cuda.device_count()
    gpu_name  = torch.cuda.get_device_name(0)

    # P100 not supported by current PyTorch – fall back to CPU
    if "P100" in gpu_name:
        device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
        print(f"GPU: {gpu_name} (not supported by PyTorch) – using CPU mode")
    elif num_gpus > 1:
        device      = "cuda"
        BATCH_SIZE  = 128   # 32 GB total VRAM (T4×2)  – dense output is larger than CLS only
        NUM_WORKERS = 4
        print(f"GPU: {gpu_name} | Count: {num_gpus} | Batch: {BATCH_SIZE}")
    else:
        device      = "cuda"
        BATCH_SIZE  = 96    # T4 single – reduced from 192 because we keep all patch tokens
        NUM_WORKERS = 2
        print(f"GPU: {gpu_name} | Count: {num_gpus} | Batch: {BATCH_SIZE}")
else:
    device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
    print("CPU mode")

In [ ]:
# ---------------------------------------------------------------------------
# Dataset loader – identical to the CLS notebook
# ---------------------------------------------------------------------------
class FastDataset(Dataset):
    def __init__(self, paths, proc):
        self.paths, self.proc = paths, proc

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), 0)

        inputs = self.proc(images=img, return_tensors="pt")
        inputs_squeezed = {k: v.squeeze(0) for k, v in inputs.items()}
        return inputs_squeezed, self.paths[i]


imgs = [
    (str(p.relative_to(DATASET_PATH)), str(p))
    for p in Path(DATASET_PATH).rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
]
image_ids, image_paths = [i for i, _ in imgs], [p for _, p in imgs]
print(f"Found {len(image_ids)} images")

In [ ]:
# ---------------------------------------------------------------------------
# Load model
# ---------------------------------------------------------------------------
print(f"Loading {MODEL_ID}...")
t0 = time.time()

processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=hf_token)

dtype   = torch.float16 if device == "cuda" else torch.float32
use_amp = (device == "cuda")
if use_amp:
    print("  Using FP16 for speedup.")

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    token=hf_token,
)

torch.cuda.empty_cache()

# ---------------------------------------------------------------------------
# Dense-feature wrapper
#
# Strategy: extract patch tokens (last_hidden_state[:, 1:, :]) and mean-pool
# them across the spatial dimension.  This produces a 768-D vector that:
#   • captures LOCAL texture / material / style information
#   • is LESS dominated by the global object class (unlike the [CLS] token)
#
# Reference: Oquab et al. 2023, DINOv2 §3.2 – dense tasks use patch features.
# ---------------------------------------------------------------------------
class DenseWrapper(torch.nn.Module):
    """Returns mean-pooled patch tokens (dense / texture embedding)."""

    def __init__(self, base_model):
        super().__init__()
        self.model = base_model

    def forward(self, **kwargs):
        outputs = self.model(**kwargs)
        # last_hidden_state shape: (B, 1 + num_patches, hidden_dim)
        # Index 0 = [CLS] token  –  skip it; indices 1: = patch tokens
        patch_tokens = outputs.last_hidden_state[:, 1:, :]  # (B, P, D)
        # Spatial mean-pool → (B, D)  →  same shape as the CLS embedding
        dense_emb = patch_tokens.mean(dim=1)                # (B, D)
        return dense_emb


wrapped_model = DenseWrapper(model)

if device == "cuda" and torch.cuda.device_count() > 1:
    final_model = torch.nn.DataParallel(wrapped_model).cuda()
    print("  ✓ Applied DataParallel")
else:
    final_model = wrapped_model.to(device)

final_model.eval()
if device == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"Model ready in {time.time() - t0:.1f}s")

In [ ]:
# ---------------------------------------------------------------------------
# Inference loop
# ---------------------------------------------------------------------------
loader = DataLoader(
    FastDataset(image_paths, processor),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)

all_emb, t0 = [], time.time()

with torch.no_grad():
    for inputs, _ in tqdm(loader, desc="Dense inference"):
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

        if use_amp:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                dense_emb = final_model(**inputs)          # (B, 768)
                feat      = F.normalize(dense_emb, p=2, dim=-1)
        else:
            dense_emb = final_model(**inputs)
            feat      = F.normalize(dense_emb, p=2, dim=-1)

        all_emb.append(feat.cpu().float().numpy())         # always store fp32

if device == "cuda":
    torch.cuda.synchronize()

inf_time   = time.time() - t0
embeddings = np.vstack(all_emb)                            # (N, 768)
print(f"Done: {inf_time:.1f}s | {len(image_paths) / inf_time:.1f} img/s")
print(f"Embedding matrix shape: {embeddings.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Save HDF5 + FAISS index
#
# Output convention (matches kaggle_pipeline.py download_outputs()):
#   embeddings.hdf5  →  dinov3_dense_embeddings_<ts>.hdf5
#   faiss_index.pkl  →  dinov3_dense_faiss_index_<ts>.pkl
# ---------------------------------------------------------------------------
hdf5_path  = OUTPUT_DIR + "/embeddings.hdf5"
faiss_path = OUTPUT_DIR + "/faiss_index.pkl"

with h5py.File(hdf5_path, "w") as f:
    f.create_dataset("embeddings",   data=embeddings)
    f.create_dataset("image_ids",    data=[s.encode("utf-8") for s in image_ids])
    f.create_dataset("image_paths",  data=[s.encode("utf-8") for s in image_paths])
    # Store metadata so the app can verify which feature type this is
    f.attrs["model_id"]       = MODEL_ID
    f.attrs["feature_type"]   = "dense_patch_mean_pool"
    f.attrs["embedding_dim"]  = EMBEDDING_DIM
    f.attrs["num_images"]     = len(image_ids)

print(f"✓ Saved HDF5  → {hdf5_path}")

# FAISS IndexFlatIP = cosine similarity (vectors are L2-normalised)
index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings.astype(np.float32))

with open(faiss_path, "wb") as f:
    pickle.dump(index, f)

print(f"✓ Saved FAISS → {faiss_path}")
print(f"  Total vectors indexed: {index.ntotal}")